# Cost Aware Pit Spy Gate

- Experiment slug: `cost-aware-pit-spy-gate`
- Problem type: `regression`
- Primary workflow: `boosting`
- Metrics to optimize:
- `sharpe_diff_ci_low`
- `candidate_sharpe`
- `information_ratio`
- Notebook path: `notebook/`
- Validation: 5-fold CV
- Expected close-out: metrics table plus champion model


## Experiment metadata

In [ ]:
from pathlib import Path

EXPERIMENT_SLUG = "cost-aware-pit-spy-gate"
DATASET_PATH = Path("data/runs/phase2-source-20260424")
TARGET_COLUMN = "fwd_return_5d"
PRIMARY_METRIC = "sharpe_diff_ci_low"
SECONDARY_METRICS = ['candidate_sharpe', 'information_ratio']
PROBLEM_TYPE = "regression"
RUN_MODE = "boosting"
N_SPLITS = 5
RANDOM_STATE = 42


## Reproducible setup

- Keep `N_SPLITS = 5`.
- Define the winning direction of the primary metric before comparing models.
- If you change the validation strategy, justify it in markdown.


In [ ]:
import random

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)


## Dataset loading and target definition

Load the dataset, inspect shape and dtypes, and confirm the target column.

In [ ]:
df = pd.read_csv(DATASET_PATH)  # Replace with the right loader for parquet/feather/etc.
df.head()


## Baseline

Build the first simple baseline before feature engineering so later gains are measurable.

In [ ]:
# TODO: add baseline training and 5-fold evaluation


## Feature engineering log

Document every feature block you add, why it may help, and whether it improved the metric.

In [ ]:
# TODO: build and compare feature blocks


## Experiment tracker

Use the helper below to log every run, not just the winner.

In [ ]:
experiment_rows = []


def register_experiment(
    name,
    family,
    primary_metric,
    fold_scores,
    *,
    secondary_metrics=None,
    feature_blocks=None,
    params=None,
    notes="",
):
    secondary_metrics = secondary_metrics or {}
    feature_blocks = feature_blocks or []
    params = params or {}
    experiment_rows.append(
        {
            "name": name,
            "family": family,
            "primary_metric": primary_metric,
            "cv_mean": float(np.mean(fold_scores)),
            "cv_std": float(np.std(fold_scores)),
            "fold_scores": list(fold_scores),
            "secondary_metrics": secondary_metrics,
            "feature_blocks": feature_blocks,
            "params": params,
            "notes": notes,
        }
    )


def leaderboard(ascending=True):
    board = pd.DataFrame(experiment_rows)
    if board.empty:
        return board
    return board.sort_values(["cv_mean", "cv_std"], ascending=[ascending, True]).reset_index(drop=True)


## CatBoost-first boosting experiments

Start with CatBoost. Add alternative boosters only when they create a meaningful comparison.

In [ ]:
# TODO: add CatBoost CV runs and call register_experiment(...)


## Alternative boosters

Use this section for XGBoost, LightGBM, HistGradientBoosting, or other justified baselines.

In [ ]:
# TODO: add other boosting experiments if needed


## PyTorch experiments

Use PyTorch and target `mps` first. If `mps` is unavailable, explain the fallback in markdown.

In [ ]:
try:
    import torch
except ImportError:
    torch = None

if torch is None:
    device = "missing-torch"
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Selected device: {device}")


In [ ]:
# TODO: add PyTorch model, training loop, validation, and register_experiment(...)


## Leaderboard and error analysis

Review the full experiment table, not just the best score. Explain failures and regressions.

In [ ]:
leaderboard()


## Champion model summary

Close every run with the winning model, chosen metrics, CV detail, and the next best experiment.

In [ ]:
# Winner summary

winner_name = "TODO"
winner_family = "TODO"
winner_score = "TODO"
winner_secondary_metrics = "TODO"
winner_feature_blocks = "TODO"
winner_params = "TODO"
winner_reason = "TODO"
next_iteration = "TODO"

print(f"Winner model: {winner_name}")
print(f"Family: {winner_family}")
print(f"Primary metric: {PRIMARY_METRIC} = {winner_score}")
print(f"Secondary metrics: {winner_secondary_metrics}")
print(f"Key features: {winner_feature_blocks}")
print(f"Key params: {winner_params}")
print(f"Why it won: {winner_reason}")
print(f"Next iteration: {next_iteration}")


## Results captured on 2026-04-28

Primary metric: `sharpe_diff_ci_low`; the promotion gate requires it to be positive. Secondary metrics: candidate Sharpe, SPY-relative information ratio, ending value versus same-deposit SPY, and total commissions.

Broad PIT production-economics contract: `max_assets=100` point-in-time by `dollar_volume_21d`, `max_rebalances=241`, `initial_portfolio_value=1000`, `monthly_deposit_amount=100`, `commission_rate=0.02`, `lambda_turnover=5.0`, `rebalance_on_deposit_day=true`, 5-day horizon, 236 aligned SPY observations.

| Rank | Candidate / setting | Status | Sharpe | SPY Sharpe | CI low | IR | Ending value | SPY ending | Commissions |
| ---: | --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 1 | `lightgbm_return_zscore`, max weight 0.24, band 0.04 | provisional | 1.686490 | 1.139604 | -0.078446 | 1.361028 | $19,607.92 | $10,045.59 | $416.58 |
| 2 | `lightgbm_return_zscore`, max weight 0.25, band 0.04 | provisional | 1.681303 | 1.139604 | -0.081831 | 1.350918 | $19,590.57 | $10,045.59 | $416.17 |
| 3 | `lightgbm_return_zscore`, base optimizer | provisional | 1.607539 | 1.139604 | -0.164219 | 1.310368 | $19,514.99 | $10,045.59 | $371.67 |
| 4 | `catboost_return_zscore` | provisional | 1.296346 | 1.139604 | -0.638166 | 0.883926 | $16,997.49 | $10,045.59 | $498.03 |
| 5 | `catboost_momentum_return_zscore` | provisional | 1.275287 | 1.139604 | -0.666697 | 0.831108 | $15,514.11 | $10,045.59 | $330.03 |

PyTorch MLP with `mps` available was tested on the 48-rebalance PIT screen. Best neural screen: `torch_mlp_return_deep_zscore`, Sharpe 1.559 versus SPY 1.320, CI low -0.792, ending value $7,189.50 versus SPY $6,115.11. It did not justify broad confirmation.


## Experiment close

Winner model: `lightgbm_return_zscore` with `optimizer_max_weight=0.24`, `lambda_turnover=5.0`, `no_trade_band=0.04`

Family: LightGBM regression

Primary metric: `sharpe_diff_ci_low = -0.078446` on the broad PIT production-economics validation

Secondary metrics: Sharpe 1.686490 versus SPY 1.139604, SPY-relative IR 1.361028, ending value $19,607.92 versus same-deposit SPY $10,045.59

CV detail: the repository validation is walk-forward PIT backtesting rather than random 5-fold CV because random folds would leak time-series information. The 48-rebalance screen was used for model family selection; the winner was broad-confirmed over 236 aligned SPY observations.

Key features: default Phase 2 momentum, risk, drawdown, moving-average, liquidity, volume, return, and excess-return features.

Key params: LightGBM default return-regression candidate, optimizer max weight 0.24, risk aversion 10, turnover penalty 5.0, no-trade band 0.04, 2% commission, monthly $100 deposits.

Notebook: `notebook/cost-aware-pit-spy-gate-20260428.ipynb`

Next iteration: do not promote as a confirmed SPY beat. The next best work is historical constituent membership / survivorship-bias removal and longer out-of-sample validation.


## PyTorch Sequence Model Screen - 2026-04-28

Added first-class PyTorch LSTM and Transformer autoresearch candidates using point-in-time per-ticker trailing feature windows. Device selection uses `mps` on Apple Silicon when available through `uv run --extra pytorch`.

Primary metric: candidate Sharpe versus SPY Sharpe under the production-economics PIT contract. Secondary metrics: SPY-relative information ratio, bootstrap Sharpe-difference CI low, ending value, and commission drag.

| Candidate / setting | Scope | Status | Sharpe | SPY Sharpe | IR | CI Low | Ending | SPY Ending |
| --- | --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |
| `lightgbm_return_zscore`, max weight 0.24, band 0.04 | broad 241 | provisional | 1.686490 | 1.139604 | 1.361028 | -0.078446 | $19,607.92 | $10,045.59 |
| `torch_lstm_momentum_return_zscore`, max weight 0.24, band 0.04 | broad 241 | provisional | 1.375979 | 1.139604 | 0.851981 | -0.333648 | $14,371.04 | $10,045.59 |
| `lightgbm_return_zscore`, max weight 0.24, band 0.04 | fast 48 | provisional | 2.097144 | 1.320164 | 1.593883 | -0.428909 | $7,179.08 | $6,115.11 |
| `torch_lstm_momentum_return_zscore`, max weight 0.24, band 0.04 | fast 48 | provisional | 1.635387 | 1.320164 | 0.801520 | -0.709346 | $6,982.64 | $6,115.11 |
| `torch_transformer_return_zscore`, max weight 0.24, band 0.04 | fast 48 | rejected | 1.240219 | 1.320164 | 0.201597 | -0.976523 | $6,740.75 | $6,115.11 |
| `torch_transformer_return_deep_zscore`, max weight 0.24, band 0.04 | fast 48 | rejected | 1.005355 | 1.320164 | -0.107549 | -0.988385 | $6,434.21 | $6,115.11 |
| `torch_transformer_momentum_return_zscore`, max weight 0.24, band 0.04 | fast 48 | rejected | 1.023103 | 1.320164 | -0.034211 | -0.963996 | $6,521.72 | $6,115.11 |

Winner model: `lightgbm_return_zscore`, max weight 0.24, no-trade band 0.04.
Family: LightGBM tabular regression.
Primary metric: broad PIT Sharpe = 1.686490 versus SPY = 1.139604.
Secondary metrics: IR = 1.361028, CI low = -0.078446, ending value = $19,607.92 versus SPY = $10,045.59.
CV detail: walk-forward point-in-time backtest with 236 aligned SPY observations; no random K-fold CV because this is a time-series portfolio validation where fold shuffling would leak time.
Key features: default momentum, rank, volatility, drawdown, moving-average ratio, return, liquidity, and volume z-score feature set.
Key params: `optimizer_max_weight=0.24`, `lambda_turnover=5.0`, `commission_rate=0.02`, `monthly_deposit_amount=100`, `no_trade_band=0.04`.
Notebook: `notebook/cost-aware-pit-spy-gate-20260428.ipynb`.
Next iteration: do not promote LSTM/Transformer; prioritize historical constituent membership and longer out-of-sample validation before adding model complexity.
